In [36]:
import fastf1
import pandas as pd

In [40]:
fastf1.Cache.enable_cache("cache")  # adjust path

NotADirectoryError: Cache directory does not exist! Please check for typos or create it first.

In [18]:
session = fastf1.get_session(year=2025, gp="brazil", identifier="R")
session.load()

core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 4 completed the race distance 00:00.010000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['4', '12', '1', '63', '81', '87', '30', '6', '27', '10', '23', '3

In [19]:
# Get lap-by-lap positions
laps = session.laps

# Convert to a convenient structure:
# lap_position[lap][driver] = race_position
lap_position = {}

for _, lap in laps.iterrows():
    lap_number = int(lap['LapNumber'])
    driver = lap['Driver']
    position = lap['Position']

    lap_position.setdefault(lap_number, {})
    lap_position[lap_number][driver] = position

In [20]:
max_lap = max(lap_position.keys())

In [21]:
overtakes = []
for lap in range(1, max_lap):  # compare lap L and L+1
    current = lap_position.get(lap, {})
    nxt = lap_position.get(lap + 1, {})

    # For every driver pair, check if order swapped
    for drv_a, pos_a in current.items():
        # find which driver was directly ahead of drv_a on lap L
        for drv_b, pos_b in current.items():
            if pos_b == pos_a - 1:  # drv_b was ahead of drv_a
                # Now compare positions on next lap
                next_pos_a = nxt.get(drv_a)
                next_pos_b = nxt.get(drv_b)

                if next_pos_a is None or next_pos_b is None:
                    continue

                # Position swap happened: A ahead of B
                if next_pos_a < next_pos_b:
                    # EXCLUSION FILTERS

                    # → True overtake
                    overtakes.append({
                        "lap": lap,
                        "attacker": drv_a,
                        "defender": drv_b
                    })

In [22]:
df_overtakes = pd.DataFrame(overtakes)
print(df_overtakes)

     lap attacker defender
0      1      VER      HAM
1      1      STR      TSU
2      1      SAI      HUL
3      1      BEA      GAS
4      2      VER      TSU
..   ...      ...      ...
115   59      ALB      ALO
116   62      VER      RUS
117   66      OCO      SAI
118   69      ALB      OCO
119   69      HAD      HUL

[120 rows x 3 columns]
